# Notebook 00 — Extracción y enriquecimiento del dataset

Este notebook construye el pipeline de datos del proyecto.

Objetivo: ampliar el dataset original de 1000 comentarios
a ~5000 comentarios reales de YouTube para reducir
el overfitting del modelo de clasificación.

Pipeline:
1. Conectar a YouTube API y extraer comentarios reales
2. Etiquetar cada comentario con Detoxify
3. Combinar con el dataset original
4. Guardar dataset_enriquecido.csv

In [12]:
# Manejo de datos
import pandas as pd
import re
import time
import os

# YouTube API
from googleapiclient.discovery import build

# Detoxify para etiquetar toxicidad
from detoxify import Detoxify

# Cargar variables de entorno
from dotenv import load_dotenv

# Cargamos las variables del archivo .env
load_dotenv()

True

# 2. Configuración

Cargamos la API key de YouTube desde el archivo .env
y definimos los vídeos de los que extraeremos comentarios.

In [13]:
# Leemos la API key del archivo .env
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

# Comprobamos que se cargó correctamente
if YOUTUBE_API_KEY:
    print("✅ API key cargada correctamente")
    print(f"   Primeros 8 caracteres: {YOUTUBE_API_KEY[:8]}...")
else:
    print("❌ Error: no se encontró YOUTUBE_API_KEY en el archivo .env")

✅ API key cargada correctamente
   Primeros 8 caracteres: AIzaSyB-...


In [14]:
# Leemos la API key del archivo .env
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

# Comprobamos que se cargó correctamente
if YOUTUBE_API_KEY:
    print("✅ API key cargada correctamente")
    print(f"   Primeros 8 caracteres: {YOUTUBE_API_KEY[:8]}...")
else:
    print("❌ Error: no se encontró YOUTUBE_API_KEY en el archivo .env")

# Vídeos de YouTube verificados con comentarios activos
VIDEOS = [
    "5YFVYrZyg5U",
    "FBpPSVQHSmk",
    "cwSXDr7XkNc",
    "o5yKHBaRn8c",
    "7TEnJ5pyFDg",
    "HiyzzcuaAac",
    "oo9c9HC-pmM",
    "iG9CE55wbtY",
    "Lr6dJ4jmIYc",
    "1srQ7Mq_ToI"
]

# Cuántos comentarios queremos por vídeo
COMENTARIOS_POR_VIDEO = 500

print(f"Vídeos configurados: {len(VIDEOS)}")
print(f"Comentarios por vídeo: {COMENTARIOS_POR_VIDEO}")
print(f"Total esperado: ~{len(VIDEOS) * COMENTARIOS_POR_VIDEO} comentarios")

✅ API key cargada correctamente
   Primeros 8 caracteres: AIzaSyB-...
Vídeos configurados: 10
Comentarios por vídeo: 500
Total esperado: ~5000 comentarios


# 3. Conexión a YouTube API

Creamos el cliente que nos permite comunicarnos
con YouTube desde Python.

In [15]:
# Creamos el cliente de YouTube API
youtube = build(
    "youtube",   # nombre del servicio
    "v3",        # versión de la API
    developerKey=YOUTUBE_API_KEY
)

print("✅ Conexión a YouTube API establecida")

✅ Conexión a YouTube API establecida


# 4. Extracción de comentarios de YouTube

Definimos la función que extrae comentarios
de un vídeo concreto usando su ID.

In [16]:
def extraer_comentarios(youtube, video_id, max_comentarios=500):
    """
    Extrae comentarios de un vídeo de YouTube.

    Parámetros:
        youtube: cliente de YouTube API
        video_id: ID del vídeo (la parte después de ?v= en la URL)
        max_comentarios: máximo de comentarios a extraer

    Devuelve:
        Lista de textos de comentarios limpios
    """
    comentarios = []

    try:
        # Hacemos la petición a YouTube
        peticion = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(max_comentarios, 100),
            textFormat="plainText",
            order="relevance"
        )

        # Mientras haya páginas de comentarios y no hayamos llegado al límite
        while peticion and len(comentarios) < max_comentarios:

            # Ejecutamos la petición y obtenemos la respuesta
            respuesta = peticion.execute()

            # Recorremos cada comentario de la respuesta
            for item in respuesta.get("items", []):

                # Extraemos el texto del comentario
                texto = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]

                # Limpieza básica
                texto = re.sub(r"http\S+", "", texto)   # eliminar URLs
                texto = re.sub(r"@\w+", "", texto)       # eliminar menciones
                texto = texto.strip()                     # eliminar espacios

                # Solo guardamos comentarios con suficiente contenido
                if len(texto) > 10:
                    comentarios.append(texto)

            # Comprobamos si hay más páginas de comentarios
            if "nextPageToken" in respuesta:
                peticion = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=min(max_comentarios - len(comentarios), 100),
                    textFormat="plainText",
                    order="relevance",
                    pageToken=respuesta["nextPageToken"]
                )
            else:
                # No hay más páginas, terminamos
                peticion = None

    except Exception as error:
        print(f"   ⚠️ Error en vídeo {video_id}: {str(error)[:80]}")

    return comentarios

# 5. Extracción de comentarios de todos los vídeos

Aplicamos la función a cada vídeo de nuestra lista
y guardamos todos los comentarios en una lista común.

In [17]:
# Lista donde guardaremos todos los comentarios extraídos
todos_los_comentarios = []

print("Iniciando extracción de comentarios...")
print("=" * 50)

# Recorremos cada vídeo de la lista
for i, video_id in enumerate(VIDEOS):

    print(f"\nVídeo {i+1}/{len(VIDEOS)}: {video_id}")

    # Extraemos los comentarios de este vídeo
    comentarios_video = extraer_comentarios(
        youtube,
        video_id,
        max_comentarios=COMENTARIOS_POR_VIDEO
    )

    # Los añadimos a la lista general
    todos_los_comentarios.extend(comentarios_video)

    print(f"   ✅ Extraídos: {len(comentarios_video)} comentarios")
    print(f"   📊 Total acumulado: {len(todos_los_comentarios)}")

    # Pausa entre vídeos para respetar los límites de la API
    time.sleep(1)

print("\n" + "=" * 50)
print(f"✅ Extracción completada")
print(f"📊 Total comentarios extraídos: {len(todos_los_comentarios)}")

Iniciando extracción de comentarios...

Vídeo 1/10: 5YFVYrZyg5U


   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 500

Vídeo 2/10: FBpPSVQHSmk
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 1000

Vídeo 3/10: cwSXDr7XkNc
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 1500

Vídeo 4/10: o5yKHBaRn8c
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 2000

Vídeo 5/10: 7TEnJ5pyFDg
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 2500

Vídeo 6/10: HiyzzcuaAac
   ✅ Extraídos: 471 comentarios
   📊 Total acumulado: 2971

Vídeo 7/10: oo9c9HC-pmM
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 3471

Vídeo 8/10: iG9CE55wbtY
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 3971

Vídeo 9/10: Lr6dJ4jmIYc
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 4471

Vídeo 10/10: 1srQ7Mq_ToI
   ✅ Extraídos: 500 comentarios
   📊 Total acumulado: 4971

✅ Extracción completada
📊 Total comentarios extraídos: 4971


# 6. Etiquetado de toxicidad con Detoxify

Analizamos cada comentario extraído de YouTube
y le asignamos una etiqueta de toxicidad.

Detoxify devuelve una puntuación entre 0 y 1.
Usamos un umbral de 0.5:
- Puntuación >= 0.5 → True (tóxico)
- Puntuación <  0.5 → False (no tóxico)

In [18]:
# Cargamos el modelo de Detoxify
# La primera vez descarga el modelo automáticamente
print("Cargando modelo Detoxify...")
modelo_toxicidad = Detoxify('original')
print("✅ Modelo cargado correctamente")

Cargando modelo Detoxify...
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /home/mar/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:31<00:00, 14.1MB/s] 
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 32843.32it/s]


✅ Modelo cargado correctamente


# 7. Aplicando Detoxify a los comentarios extraídos

Analizamos los 4971 comentarios uno a uno.
Cada comentario recibe una puntuación de toxicidad
y una etiqueta True o False.

Este proceso puede tardar varios minutos.

In [19]:
# Umbral para decidir si un comentario es tóxico
UMBRAL_TOXICIDAD = 0.5

# Listas donde guardaremos los resultados
textos_etiquetados  = []
etiquetas           = []
puntuaciones        = []

total = len(todos_los_comentarios)
print(f"Etiquetando {total} comentarios...")
print(f"Umbral de toxicidad: {UMBRAL_TOXICIDAD}")
print("=" * 50)

# Procesamos en lotes para ser más eficientes
TAMANO_LOTE = 32

for i in range(0, total, TAMANO_LOTE):

    # Cogemos el lote actual
    lote = todos_los_comentarios[i:i + TAMANO_LOTE]

    # Detoxify analiza todo el lote de una vez
    resultados = modelo_toxicidad.predict(lote)

    # Procesamos cada resultado del lote
    for j, texto in enumerate(lote):
        puntuacion = resultados['toxicity'][j]
        es_toxico  = bool(puntuacion >= UMBRAL_TOXICIDAD)

        textos_etiquetados.append(texto)
        etiquetas.append(es_toxico)
        puntuaciones.append(round(float(puntuacion), 4))

    # Mostramos progreso cada 500 comentarios
    procesados = min(i + TAMANO_LOTE, total)
    if procesados % 500 == 0 or procesados == total:
        print(f"  Procesados: {procesados}/{total} "
              f"({procesados/total*100:.1f}%)")

print("\n✅ Etiquetado completado")
print(f"Tóxicos:     {sum(etiquetas)} "
      f"({sum(etiquetas)/len(etiquetas)*100:.1f}%)")
print(f"No tóxicos:  {len(etiquetas) - sum(etiquetas)} "
      f"({(len(etiquetas)-sum(etiquetas))/len(etiquetas)*100:.1f}%)")

Etiquetando 4971 comentarios...
Umbral de toxicidad: 0.5
  Procesados: 4000/4971 (80.5%)
  Procesados: 4971/4971 (100.0%)

✅ Etiquetado completado
Tóxicos:     146 (2.9%)
No tóxicos:  4825 (97.1%)


In [20]:
# Analizamos la distribución con distintos umbrales
import numpy as np

umbrales = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

print("=" * 55)
print("ANÁLISIS DE DISTRIBUCIÓN POR UMBRAL")
print("=" * 55)
print(f"{'Umbral':>8} {'Tóxicos':>10} {'No tóxicos':>12} {'% tóxico':>10}")
print("-" * 55)

for umbral in umbrales:
    toxicos    = sum(1 for p in puntuaciones if p >= umbral)
    no_toxicos = len(puntuaciones) - toxicos
    porcentaje = toxicos / len(puntuaciones) * 100
    equilibrio = "✅" if 30 <= porcentaje <= 55 else "⚠️"
    print(f"{umbral:>8} {toxicos:>10} {no_toxicos:>12} "
          f"{porcentaje:>9.1f}% {equilibrio}")

print("=" * 55)
print("✅ = distribución equilibrada (30-55% tóxicos)")

ANÁLISIS DE DISTRIBUCIÓN POR UMBRAL
  Umbral    Tóxicos   No tóxicos   % tóxico
-------------------------------------------------------
     0.1        506         4465      10.2% ⚠️
    0.15        412         4559       8.3% ⚠️
     0.2        342         4629       6.9% ⚠️
    0.25        273         4698       5.5% ⚠️
     0.3        238         4733       4.8% ⚠️
     0.4        190         4781       3.8% ⚠️
     0.5        146         4825       2.9% ⚠️
✅ = distribución equilibrada (30-55% tóxicos)


# 8. Descarga de dataset externo de Kaggle

Descargamos el dataset "Jigsaw Toxic Comment Classification"
que contiene 150.000 comentarios etiquetados por humanos.

Usaremos solo los comentarios tóxicos para equilibrar
nuestro dataset con la clase que nos falta.

In [21]:
import kagglehub
import os

# Descargamos el dataset de Kaggle
print("Descargando dataset de Kaggle...")
print("Esto puede tardar unos minutos...")

ruta_kaggle = kagglehub.dataset_download(
    "julian3833/jigsaw-toxic-comment-classification-challenge"
)

print(f"\n✅ Dataset descargado en: {ruta_kaggle}")

# Listamos los archivos disponibles
archivos = os.listdir(ruta_kaggle)
print(f"\nArchivos disponibles:")
for archivo in archivos:
    print(f"  → {archivo}")

Descargando dataset de Kaggle...
Esto puede tardar unos minutos...


100%|██████████| 53.4M/53.4M [00:03<00:00, 15.0MB/s]

Extracting files...



✅ Dataset descargado en: /home/mar/.cache/kagglehub/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/versions/1

Archivos disponibles:
  → train.csv
  → sample_submission.csv
  → test.csv
  → test_labels.csv


# 9. Exploración del dataset de Kaggle

Cargamos el archivo train.csv y exploramos
su estructura para entender qué tenemos.

In [22]:
# Cargamos el dataset de Kaggle
ruta_train = os.path.join(ruta_kaggle, "train.csv")
df_kaggle = pd.read_csv(ruta_train)

# Exploramos la estructura
print(f"Dimensiones: {df_kaggle.shape}")
print(f"\nColumnas: {list(df_kaggle.columns)}")
print(f"\nPrimeras 3 filas:")
df_kaggle.head(3)

Dimensiones: (159571, 8)

Columnas: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Primeras 3 filas:


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0


# 10. Preparación de comentarios tóxicos de Kaggle

Filtramos solo los comentarios tóxicos del dataset
de Kaggle para equilibrar nuestra clase minoritaria.

Estrategia:
- Cogemos comentarios tóxicos de Kaggle
- Los combinamos con nuestros comentarios de YouTube
- Resultado: dataset equilibrado con más ejemplos

In [23]:
# Vemos la distribución actual del dataset de Kaggle
toxicos_kaggle    = df_kaggle['toxic'].sum()
no_toxicos_kaggle = len(df_kaggle) - toxicos_kaggle

print("=" * 45)
print("DISTRIBUCIÓN DATASET KAGGLE")
print("=" * 45)
print(f"Total comentarios:  {len(df_kaggle):>8}")
print(f"Tóxicos (toxic=1):  {toxicos_kaggle:>8} "
      f"({toxicos_kaggle/len(df_kaggle)*100:.1f}%)")
print(f"No tóxicos:         {no_toxicos_kaggle:>8} "
      f"({no_toxicos_kaggle/len(df_kaggle)*100:.1f}%)")
print("=" * 45)

DISTRIBUCIÓN DATASET KAGGLE
Total comentarios:    159571
Tóxicos (toxic=1):     15294 (9.6%)
No tóxicos:           144277 (90.4%)


# 11. Construcción del dataset final

Combinamos tres fuentes de datos:
1. Dataset original (1000 comentarios etiquetados)
2. Comentarios de YouTube etiquetados con Detoxify
3. Comentarios tóxicos de Kaggle etiquetados por humanos

Objetivo: dataset equilibrado de ~10.000 comentarios

In [24]:
# ── FUENTE 1: Dataset original ──────────────────────
df_original = pd.read_csv(
    '../../data/raw/youtoxic_english_1000.csv'
)

# Nos quedamos solo con las columnas que necesitamos
df_original = df_original[['Text', 'IsToxic']].copy()

# Renombramos para que coincidan con el resto
df_original.columns = ['Text', 'IsToxic']

print(f"Fuente 1 — Original:")
print(f"  Total:      {len(df_original)}")
print(f"  Tóxicos:    {df_original['IsToxic'].sum()}")
print(f"  No tóxicos: {(~df_original['IsToxic']).sum()}")

# ── FUENTE 2: Comentarios de YouTube ─────────────────
df_youtube = pd.DataFrame({
    'Text':    textos_etiquetados,
    'IsToxic': etiquetas
})

print(f"\nFuente 2 — YouTube + Detoxify:")
print(f"  Total:      {len(df_youtube)}")
print(f"  Tóxicos:    {df_youtube['IsToxic'].sum()}")
print(f"  No tóxicos: {(~df_youtube['IsToxic']).sum()}")

# ── FUENTE 3: Comentarios tóxicos de Kaggle ──────────

# Filtramos solo los tóxicos
df_kaggle_toxicos = df_kaggle[
    df_kaggle['toxic'] == 1
][['comment_text']].copy()

# Cogemos una muestra de 4500
df_kaggle_toxicos = df_kaggle_toxicos.sample(
    n=4500,
    random_state=42
).reset_index(drop=True)

# Renombramos y añadimos la etiqueta
df_kaggle_toxicos.columns = ['Text']
df_kaggle_toxicos['IsToxic'] = True

print(f"\nFuente 3 — Kaggle (tóxicos):")
print(f"  Total:      {len(df_kaggle_toxicos)}")
print(f"  Tóxicos:    {df_kaggle_toxicos['IsToxic'].sum()}")
print(f"  No tóxicos: {(~df_kaggle_toxicos['IsToxic']).sum()}")

Fuente 1 — Original:
  Total:      1000
  Tóxicos:    462
  No tóxicos: 538

Fuente 2 — YouTube + Detoxify:
  Total:      4971
  Tóxicos:    146
  No tóxicos: 4825

Fuente 3 — Kaggle (tóxicos):
  Total:      4500
  Tóxicos:    4500
  No tóxicos: 0


In [25]:
# ── COMBINACIÓN FINAL ─────────────────────────────────
df_final = pd.concat(
    [df_original, df_youtube, df_kaggle_toxicos],
    ignore_index=True
)

# Eliminamos duplicados por texto
antes = len(df_final)
df_final = df_final.drop_duplicates(subset=['Text'])
eliminados = antes - len(df_final)

# Eliminamos filas con texto vacío
df_final = df_final.dropna(subset=['Text'])
df_final = df_final[df_final['Text'].str.strip() != '']

print("\n" + "=" * 50)
print("DATASET FINAL COMBINADO")
print("=" * 50)
print(f"Total comentarios:  {len(df_final)}")
print(f"Tóxicos:            {df_final['IsToxic'].sum()} "
      f"({df_final['IsToxic'].mean()*100:.1f}%)")
print(f"No tóxicos:         {(~df_final['IsToxic']).sum()} "
      f"({(~df_final['IsToxic']).mean()*100:.1f}%)")
print(f"Duplicados elim.:   {eliminados}")
print("=" * 50)


DATASET FINAL COMBINADO
Total comentarios:  10414
Tóxicos:            5105 (49.0%)
No tóxicos:         5309 (51.0%)
Duplicados elim.:   57


**Verificación del dataset final:**

Tres fuentes combinadas correctamente:
- Dataset original:  1000 comentarios ✓
- YouTube + Detoxify: 4971 comentarios ✓
- Kaggle tóxicos:    4500 comentarios ✓

Dataset final: 10.414 comentarios
Distribución: 49% tóxicos / 51% no tóxicos ✓

Distribución casi idéntica al dataset original
(46.2% / 53.8%). Sin desbalance de clases.
57 duplicados eliminados.

# 12. Guardado del dataset enriquecido

Guardamos el dataset final en disco para que
los notebooks 01, 02 y 03 puedan usarlo.

In [26]:
# Ruta de destino
ruta_salida = '../../data/raw/dataset_enriquecido.csv'

# Guardamos el dataset
df_final.to_csv(ruta_salida, index=False)

# Verificamos que se guardó correctamente
tamaño = os.path.getsize(ruta_salida) / (1024 * 1024)

print("=" * 50)
print("DATASET GUARDADO CORRECTAMENTE")
print("=" * 50)
print(f"Ruta:        {ruta_salida}")
print(f"Tamaño:      {tamaño:.2f} MB")
print(f"Filas:       {len(df_final)}")
print(f"Columnas:    {list(df_final.columns)}")
print("=" * 50)
print("\n✅ Listo para usar en notebooks 01, 02 y 03")

DATASET GUARDADO CORRECTAMENTE
Ruta:        ../../data/raw/dataset_enriquecido.csv
Tamaño:      2.11 MB
Filas:       10414
Columnas:    ['Text', 'IsToxic']

✅ Listo para usar en notebooks 01, 02 y 03


In [27]:
# Comprobamos que el CSV guardado es correcto
df_verificacion = pd.read_csv(
    '../../data/raw/dataset_enriquecido.csv'
)

print("Verificación final del archivo guardado:")
print(f"  Filas:    {len(df_verificacion)}")
print(f"  Columnas: {list(df_verificacion.columns)}")
print(f"  Tóxicos:  {df_verificacion['IsToxic'].sum()}")
print(f"  Nulos:    {df_verificacion.isnull().sum().sum()}")
print(f"\nPrimeras 3 filas:")
df_verificacion.head(3)

Verificación final del archivo guardado:
  Filas:    10414
  Columnas: ['Text', 'IsToxic']
  Tóxicos:  5105
  Nulos:    0

Primeras 3 filas:


,Text,IsToxic
0,If only people would just take a step back and...,False
1,Law enforcement is not trained to shoot to app...,True
2,\r\nDont you reckon them 'black lives matter' ...,True


# 13. Conclusiones del notebook 00

## ¿Qué se hizo en este notebook?

Se construyó un pipeline completo de extracción
y enriquecimiento de datos desde tres fuentes distintas.

---

## Las tres fuentes de datos

| Fuente | Comentarios | Tóxicos | No tóxicos | Etiquetado |
|---|---|---|---|---|
| CSV original | 1.000 | 462 (46.2%) | 538 (53.8%) | Manual |
| YouTube API | 4.971 | 146 (2.9%) | 4.825 (97.1%) | Detoxify |
| Kaggle Jigsaw | 4.500 | 4.500 (100%) | 0 | Humanos |
| **TOTAL** | **10.414** | **5.105 (49%)** | **5.309 (51%)** | **Mixto** |

---

## Dataset final: dataset_enriquecido.csv

- Filas: 10.414 comentarios
- Columnas: Text, IsToxic
- Distribución: 49% tóxicos / 51% no tóxicos
- Duplicados eliminados: 57
- Valores nulos: 0

La distribución es casi idéntica al dataset original
(46.2% / 53.8%), lo que garantiza que el modelo
no tendrá sesgo hacia ninguna clase.

---

## ¿Por qué esta estrategia?

El problema original era overfitting por dataset pequeño.
Con 10 veces más datos el modelo tiene más patrones
reales que aprender y menos probabilidad de memorizar.

Los comentarios de YouTube aportaron ejemplos no tóxicos
reales de distintos contextos y temas.

Los comentarios de Kaggle aportaron ejemplos tóxicos
etiquetados por humanos de alta calidad, que es
exactamente lo que faltaba en el dataset original.

---

## ¿Qué viene ahora?

Los notebooks 01, 02 y 03 se reejecutarán con el
nuevo dataset cambiando solo una línea en cada uno:

df = pd.read_csv('../../data/raw/dataset_enriquecido.csv')

El pipeline de preprocesamiento y el modelo son
exactamente los mismos. Con más datos de calidad
esperamos reducir el overfitting significativamente.